In [ ]:
SELECT DISTINCT
    session_status_src_name,
    session_status_src_id,
    session_status_src_sys_inst_id
FROM silver_rdm_session_status
WHERE LOWER(session_status_src_name) LIKE '%cancel%'
   OR LOWER(session_status_src_id) LIKE '%cancel%'
ORDER BY session_status_src_name;

In [ ]:
SELECT
    srv.id AS service_id,
    srv.description AS service_description,
    COUNT(*) AS row_count
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_service srv
    ON e.activity_service_id = srv.id
WHERE LOWER(srv.description) LIKE '%cancel%'
GROUP BY
    srv.id,
    srv.description
ORDER BY row_count DESC;

In [ ]:
SELECT
    st.id AS activity_status_id,
    st.description AS activity_status_description,
    COUNT(*) AS row_count
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activityheader h
    ON e.activity_header_id = h.id
LEFT JOIN silver_wip_activitystatus st
    ON h.activity_status_id = st.id
GROUP BY
    st.id,
    st.description
ORDER BY row_count DESC;

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

keywords = ["dna", "cancel", "booked", "attended", "did not attend"]

results = []

tables = [
    t.name for t in spark.catalog.listTables()
    if t.name.startswith("silver_wip_")
]

for table_name in tables:
    try:
        df = spark.table(table_name)
        string_cols = [
            field.name for field in df.schema.fields
            if isinstance(field.dataType, StringType)
        ]

        for col_name in string_cols:
            condition = None
            for kw in keywords:
                c = F.lower(F.col(col_name)).contains(kw)
                condition = c if condition is None else (condition | c)

            matches = (
                df
                .where(condition)
                .select(
                    F.lit(table_name).alias("table_name"),
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("matched_value")
                )
                .groupBy("table_name", "column_name", "matched_value")
                .count()
            )

            if matches.take(1):
                results.append(matches)

    except Exception as ex:
        print(f"Skipped {table_name}: {ex}")

if results:
    final_df = results[0]
    for r in results[1:]:
        final_df = final_df.unionByName(r)

    display(final_df.orderBy("table_name", "column_name", F.desc("count")))
else:
    print("No matches found.")

In [ ]:
schema_rows = []

tables = [
    t.name for t in spark.catalog.listTables()
    if t.name.startswith("silver_wip_")
]

for table_name in tables:
    try:
        df = spark.table(table_name)
        for field in df.schema.fields:
            schema_rows.append((table_name, field.name, str(field.dataType)))
    except Exception as ex:
        schema_rows.append((table_name, f"ERROR: {ex}", ""))

schema_df = spark.createDataFrame(
    schema_rows,
    ["table_name", "column_name", "data_type"]
)

display(schema_df.orderBy("table_name", "column_nam